# Comparative epitope mapping

Work out **where an antibody probably binds** from the fact that it binds some
species' versions of a protein and not others.

You do not need to know how to code. There are four steps, and you run each one
by pressing the **▶ play button** on the left of the grey box below it.

1. **Set up** - installs the tool (about 2 minutes, once per session)
2. **Check it works** - runs a built-in example with a known answer
3. **Enter your data** - fill in the boxes
4. **Run and read the results**

> **Before you start:** go to **Runtime → Change runtime type** and make sure it
> says CPU. You do not need a GPU.

---

## Step 1 - Set up

The code lives in a **private** GitHub repository, so this notebook cannot
download it on its own. Get a copy first:

1. Open <https://github.com/NanoLlama/epitope_align> in another tab
2. Switch the branch dropdown to `claude/antibody-epitope-identification-ervnyf`
3. Click the green **Code** button → **Download ZIP**

Then press play below and choose that ZIP file when the **Choose Files** button
appears.

*(If the repository is ever made public, replace this cell's code with the
one-liner in the comment at the top of it.)*


In [ ]:
# If the repo were public, this whole cell could be replaced by:
#   !pip install -q git+https://github.com/NanoLlama/epitope_align.git

import glob, os, subprocess, sys, zipfile

print('Installing MAFFT and DSSP (better alignments and secondary structure)...')
subprocess.run('apt-get -qq update', shell=True, capture_output=True)
subprocess.run('apt-get -qq install -y mafft dssp', shell=True, capture_output=True)

if not glob.glob('/content/epitope_align*/pyproject.toml'):
    from google.colab import files
    print('\nNow choose the ZIP you downloaded from GitHub:')
    uploaded = files.upload()
    name = list(uploaded)[0]
    with zipfile.ZipFile(name) as archive:
        archive.extractall('/content')
    os.remove(name)

folder = glob.glob('/content/epitope_align*/pyproject.toml')[0].rsplit('/', 1)[0]
print(f'\nInstalling from {folder} ...')
subprocess.run(f'pip install -q -e "{folder}"', shell=True, check=True)

import epitope_map
print(f'\nReady: epitope-map {epitope_map.__version__}')
print('MAFFT:', 'yes' if subprocess.run('which mafft', shell=True, capture_output=True).returncode == 0 else 'no')
print('DSSP: ', 'yes' if subprocess.run('which mkdssp', shell=True, capture_output=True).returncode == 0 else 'no')


---

## Step 2 - Check it works

This runs a made-up example where the answer is known in advance: six invented
species, three that bind and three that do not, and an epitope deliberately
planted at residues **65, 66, 68, 70, 72, 114, 116 and 118**.

If the top patch printed below is exactly those eight residues, everything is
working.


In [ ]:
!epitope-map --demo --outdir /content/demo-run


---

## Step 3 - Enter your data

Fill in the boxes on the right, then press play. Nothing is analysed yet - this
cell only writes down what you typed.

**Sequences.** Easiest is UniProt accession numbers, one per species, written as
`label=ACCESSION` and separated by commas, e.g.
`mouse=Q61503,rat=P21590,human=P21589`. Look each one up at
<https://www.uniprot.org> by searching your protein plus the species name; the
accession is the code like `P21589` at the top of the entry page. Use whatever
labels you like - they only have to match the binding table below.

*Alternatively* set `use_uploaded_fasta` and you will be asked for a FASTA file.

**Binding.** One line per species: the same label, a comma, then `binder`,
`non_binder`, or `unknown`. Species marked `unknown` are shown but not used for
scoring. You need at least one of each of the first two, and the more species
you have the sharper the answer.

**Reference.** The species the structure belongs to. It must be one that binds -
all numbering in the results refers to it.

**Structure.** Any of:
- an AlphaFold accession like `AF-P21589-F1` (use the reference species'
  UniProt accession; every UniProt page links its AlphaFold model)
- a Protein Data Bank ID like `4H2I` if an experimental structure exists
- leave `use_uploaded_structure` ticked to upload your own `.pdb`/`.cif` file

**Ectodomain** (optional). If you only care about part of the protein - say the
bit outside the cell - enter it as `25-240` in the structure's own numbering.
Leave blank to analyse everything.


In [ ]:
# @title Your inputs { display-mode: "form" }
# @markdown ### Sequences
sequences = "mouse=REPLACE_ME,rat=REPLACE_ME,human=REPLACE_ME"  # @param {type:"string"}
use_uploaded_fasta = False  # @param {type:"boolean"}

# @markdown ### Binding results (one species per line)
binding_table = "mouse,binder\nrat,binder\nhuman,non_binder"  # @param {type:"string"}

# @markdown ### Reference species (must be one that binds)
reference = "mouse"  # @param {type:"string"}

# @markdown ### Structure of the reference species
structure = "AF-REPLACE_ME-F1"  # @param {type:"string"}
use_uploaded_structure = False  # @param {type:"boolean"}

# @markdown ### Optional settings
ectodomain = ""  # @param {type:"string"}
chain = ""  # @param {type:"string"}

import pathlib, textwrap

work = pathlib.Path('/content/my-run')
work.mkdir(parents=True, exist_ok=True)

binding_path = work / 'binding.csv'
rows = [line.strip() for line in binding_table.splitlines() if line.strip()]
binding_path.write_text('species,binding\n' + '\n'.join(rows) + '\n')

if use_uploaded_fasta:
    from google.colab import files
    print('Choose your FASTA file:')
    uploaded = files.upload()
    sequences_arg = str(work / list(uploaded)[0])
    pathlib.Path(sequences_arg).write_bytes(list(uploaded.values())[0])
else:
    sequences_arg = sequences

if use_uploaded_structure:
    from google.colab import files
    print('Choose your structure file (.pdb or .cif):')
    uploaded = files.upload()
    structure_arg = str(work / list(uploaded)[0])
    pathlib.Path(structure_arg).write_bytes(list(uploaded.values())[0])
else:
    structure_arg = structure

print('Species and binding calls:')
print(textwrap.indent(binding_path.read_text(), '    '))
print(f'reference : {reference}')
print(f'structure : {structure_arg}')
print(f'sequences : {sequences_arg}')
print(f'range     : {ectodomain or "whole chain"}')


---

## Step 4 - Run it

This fetches anything it needs, aligns the sequences, measures the surface of the
structure and ranks the candidate patches. A small protein takes under a minute.

Read the **warnings** it prints. They are not errors - they are the honest
limitations of your particular run, and the biggest one is usually that your
binders and non-binders are two separate branches of the family tree, which
leaves a lot of irrelevant differences looking meaningful.


In [ ]:
import shlex, subprocess

command = [
    'epitope-map',
    '--sequences', sequences_arg,
    '--binding', str(binding_path),
    '--reference', reference,
    '--structure', structure_arg,
    '--outdir', '/content/my-run/results',
]
if ectodomain.strip():
    command += ['--ectodomain', ectodomain.strip()]
if chain.strip():
    command += ['--chain', chain.strip()]

print(' '.join(shlex.quote(part) for part in command), '\n')
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('--- it stopped with this message ---')
    print(result.stderr)


---

## Step 5 - Read the results

The report below is the thing to read. The key sections are *How much signal is
there?* (whether to trust any of it), *Top candidate patches* (the answer), and
*Suggested next experiments* (what to make in the lab).

Remember what this is: a **ranked shortlist of hypotheses to test**, not a
prediction. The top patch being right is something your chimeras and mutants
decide, not the software.


In [ ]:
from IPython.display import Markdown, display
import pandas as pd, pathlib

results = pathlib.Path('/content/my-run/results')
display(Markdown((results / 'report.md').read_text()))


### The candidate patches, as a table


In [ ]:
patches = pd.read_csv(results / 'patches.tsv', sep='\t')
display(patches[['patch_id', 'rank_raw', 'rank_normalized', 'n_residues',
                 'residues', 'total_score', 'mean_rsa', 'spread_A', 'flags']])


### The mutants worth making

`gain_of_binding` rows are the convincing experiment: they put the binder's
residue into a species that does *not* bind, so a positive result cannot be
explained away as a badly folded protein.


In [ ]:
mutants = pd.read_csv(results / 'mutants.tsv', sep='\t')
display(mutants[['patch_id', 'direction', 'background_species', 'mutation',
                 'numbering', 'grantham', 'rsa', 'priority']].head(20))


### Download everything

Includes `session.pml`, which opens the structure in PyMOL with the candidate
patches coloured in.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/epitope-results', 'zip', results)
files.download('/content/epitope-results.zip')
